# Adjustable Robust Optimization

Static robust optimization protects against uncertainty by requiring feasibility for *every* realization before the uncertain parameters are revealed. This is often too conservative: in many practical settings some decisions can be **deferred** until after part of the uncertainty is observed.

**Adjustable robust optimization (ARO)** {cite:p}`BenTalGoryashko2004` models this two-stage structure explicitly:

| Stage | Variables | When decided | Must satisfy |
|---|---|---|---|
| 1 (here-and-now) | $x$ | Before observing $\xi$ | Constraints for *all* $\xi \in \mathcal{U}$ |
| 2 (wait-and-see) | $y(\xi)$ | After observing $\xi$ | Constraints for the *actual* $\xi$ |

The two-stage minimax problem is:

$$
\min_{x} \max_{\xi \in \mathcal{U}} \min_{y} \; f(x, y) \quad
\text{s.t.}\; g(x, y, \xi) \le 0
$$

## Affine Decision Rules

Optimizing over arbitrary functions $y(\xi)$ is generally intractable. The
**affine decision rule** (ADR) approximation {cite:p}`BenTalGoryashko2004` restricts
recourse to affine functions of the uncertainty:

$$
y(\xi) = y_0 + \sum_{j=1}^{n_\xi} Y_j \cdot \xi_j, \qquad
\xi_j = p_j - \bar{p}_j
$$

where $y_0$ (intercept) and $Y_j$ (policy columns) become ordinary decision
variables. The ADR is **optimal** for problems where the feasible set is
convex and uncertainty enters linearly {cite:p}`BenTalGoryashko2004`. For
general nonlinear recourse it is an inner approximation.

The power of affine rules over static robust optimization was studied by
{cite:t}`Bertsimas2010`: they show that for LP problems with uncertain
right-hand sides, affine rules are optimal; for uncertain constraint matrices
they can be suboptimal.

## discopt Pipeline

```
Nominal model  →  AffineDecisionRule.apply()  →  RobustCounterpart.formulate()
(has y, ξ)        (y → y₀ + ΣYⱼξⱼ;              (ξ → worst-case constant;
                   y retired from variables)        model is deterministic)
                        ↓
                  Deterministic model: optimize over (x, y₀, Y_cols)
```

In [ ]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_ENABLE_X64"] = "1"

import discopt.modeling as dm
from discopt.ro import AffineDecisionRule, BoxUncertaintySet, RobustCounterpart

## Example 1 — Two-Stage Inventory Management

A retailer places a **first-stage** order $x$ before observing demand, then can place a **second-stage** emergency order $y$ once demand $d$ is known. The emergency order is more expensive ($c_2 = 8$ vs $c_1 = 2$). Demand is uncertain: $d \in [60, 100]$ with nominal $\bar{d} = 80$.

$$
\min_{x, y(d)}\; 2x + 8y \quad
\text{s.t.}\; x + y \ge d,\; 0 \le x \le 200,\; 0 \le y \le 100
$$

We compare three approaches: nominal (ignore uncertainty), static robust (worst-case $d=100$), and adjustable robust (recourse adapts to realized demand).

In [ ]:
c1, c2 = 2.0, 8.0
d_bar, delta = 80.0, 20.0


def build(name):
    m = dm.Model(name)
    x = m.continuous("order", lb=0, ub=200)
    y = m.continuous("emergency", lb=0, ub=100)
    d = m.parameter("d", value=d_bar)
    m.minimize(c1 * x + c2 * y)
    m.subject_to(x + y >= d, name="demand")
    return m, x, y, d


# --- Nominal ---
m_nom, _, _, _ = build("nominal")
r_nom = m_nom.solve()
print(f"Nominal (d=80):    obj = {r_nom.objective:.2f},  {r_nom.x}")

# --- Static robust ---
m_stat, _, _, d_stat = build("static")
RobustCounterpart(m_stat, BoxUncertaintySet(d_stat, delta=delta)).formulate()
r_stat = m_stat.solve()
print(f"Static  (d<=100):  obj = {r_stat.objective:.2f},  {r_stat.x}")

# --- Adjustable robust ---
m_adj, x_adj, y_adj, d_adj = build("adjustable")
adr = AffineDecisionRule(y_adj, uncertain_params=d_adj)
adr.apply()
RobustCounterpart(m_adj, BoxUncertaintySet(d_adj, delta=delta)).formulate()
r_adj = m_adj.solve()
print(f"ADR     (d<=100):  obj = {r_adj.objective:.2f},  {r_adj.x}")
print(f"\nADR policy: Y0 = {r_adj.x['adr_Y0']:.4f}")
print("  Y0~0 means no adaptation (same as static)")
print("  Y0~1 would mean full adaptation to demand")

In this example, the here-and-now order $x$ is cheap (\$2) while the emergency recourse $y$ is expensive (\$8). The ADR recovers the static robust solution because the optimal strategy is to order everything upfront at the cheap price rather than wait. The ADR correctly sets $Y_0 \approx 0$ (no adaptation), confirming that recourse has no value when the first-stage option dominates on cost.

ADR provides genuine improvement when the recourse is *cheaper* than the first-stage decision, as in the next example.

## Example 2 — When Does ADR Help?

For simple two-variable LPs with a single uncertain parameter, the ADR often matches the static robust solution. This is expected: the theoretical advantage of affine rules requires structural asymmetry, such as multiple constraints with different uncertainty exposure, or integer first-stage variables {cite:p}`Bertsimas2010`.

Here we show a case where the ADR produces a non-trivial policy (Y0 > 0) but achieves the same worst-case cost, and explain why.

In [ ]:
d_bar, delta = 40.0, 10.0

# --- Static ---
m_s = dm.Model("static")
xs = m_s.continuous("x", lb=0, ub=30)
ys = m_s.continuous("y", lb=0, ub=30)
ds = m_s.parameter("d", value=d_bar)
m_s.minimize(5.0 * xs + 3.0 * ys)
m_s.subject_to(xs + ys >= ds)
RobustCounterpart(m_s, BoxUncertaintySet(ds, delta=delta)).formulate()
r_s = m_s.solve()
print(f"Static:  obj = {r_s.objective:.2f},  x = {r_s.x['x']:.1f},  y = {r_s.x['y']:.1f}")

# --- ADR ---
m_a = dm.Model("adjustable")
xa = m_a.continuous("x", lb=0, ub=30)
ya = m_a.continuous("y", lb=0, ub=30)
da = m_a.parameter("d", value=d_bar)
m_a.minimize(5.0 * xa + 3.0 * ya)
m_a.subject_to(xa + ya >= da)
adr = AffineDecisionRule(ya, uncertain_params=da)
adr.apply()
RobustCounterpart(m_a, BoxUncertaintySet(da, delta=delta)).formulate()
r_a = m_a.solve()

Y0 = r_a.x["adr_Y0"]
y0 = r_a.x["adr_intercept"]
print(f"ADR:     obj = {r_a.objective:.2f},  x = {r_a.x['x']:.1f},  y0 = {y0:.1f},  Y0 = {Y0:.3f}")
print(f"\n  y(d=30) = {y0 + Y0 * (-10):.1f},  y(d=40) = {y0:.1f},  y(d=50) = {y0 + Y0 * 10:.1f}")
print("\nBoth achieve the same objective. The ADR finds a policy where y adapts")
print("to demand, but the worst-case cost is the same because the demand")
print("constraint and objective penalty balance out in this simple structure.")

## Example 3 — Multi-Parameter Recourse

A recourse variable can respond to *multiple* uncertain parameters simultaneously.
Each uncertain parameter $p_i$ contributes one policy column $Y_i$:

$$
y(\xi) = y_0 + Y_c \cdot (c - \bar{c}) + Y_d \cdot (d - \bar{d})
$$

Here $c$ is an uncertain cost parameter and $d$ is uncertain demand.

In [ ]:
m = dm.Model("multi_param_recourse")

x = m.continuous("x", lb=0, ub=100)
y = m.continuous("y", lb=0, ub=100)

c = m.parameter("c", value=3.0)  # uncertain unit cost
d = m.parameter("d", value=40.0)  # uncertain demand

m.minimize(c * x + 2 * y)
m.subject_to(x + y >= d, name="demand")

# y adapts to both cost and demand deviations
adr = AffineDecisionRule(y, uncertain_params=[c, d], prefix="pol")
adr.apply()

# Apply static RO to both uncertain parameters
rc = RobustCounterpart(m, [BoxUncertaintySet(c, delta=0.5), BoxUncertaintySet(d, delta=10.0)])
rc.formulate()

r = m.solve()
print(f"Multi-param ADR:  obj = {r.objective:.2f}")
print(f"  x = {r.x['x']:.2f}")
print(f"  y0 = {r.x['pol_intercept']:.2f}")
print(f"  Y_cost = {r.x['pol_Y0']:.4f}  (sensitivity to cost)")
print(f"  Y_demand = {r.x['pol_Y1']:.4f}  (sensitivity to demand)")

## API Summary

```python
from discopt.ro import AffineDecisionRule, BoxUncertaintySet, RobustCounterpart

# 1. Build nominal two-stage model
m = dm.Model()
x = m.continuous("x", ...)                  # here-and-now
y = m.continuous("y", ...)                  # wait-and-see (recourse)
p = m.parameter("p", value=nominal)         # uncertain parameter
# ... build objective and constraints ...

# 2. Apply affine decision rule  y → y₀ + Y₀·(p - p̄)
adr = AffineDecisionRule(y, uncertain_params=p)   # or list of params
adr.apply()                                        # modifies m in-place

# 3. Apply static robust counterpart for remaining uncertainty
rc = RobustCounterpart(m, BoxUncertaintySet(p, delta=0.1 * nominal))
rc.formulate()                                     # modifies m in-place

# 4. Solve — decision variables are now (x, y₀, Y₀)
result = m.solve()
```

**After `apply()`:**
- `adr.intercept` — the $y_0$ variable
- `adr.policy_columns` — list of $Y_j$ variables
- `adr.affine_expression` — the full expression $y_0 + \sum_j Y_j \xi_j$
- `adr.perturbations` — list of perturbation expressions $\xi_j = p_j - \bar{p}_j$
- `adr.n_policy_columns` — total number of scalar uncertain components

## Theoretical Guarantees and Limitations

**Optimality** {cite:t}`BenTalGoryashko2004`: For LP problems with
uncertainty appearing only in the right-hand side, affine decision rules are
*optimal* — no other measurable recourse policy achieves a lower cost.

**Sub-optimality** {cite:t}`Bertsimas2010`: For uncertain constraint matrices
(i.e., $A(\xi)x \le b(\xi)$), affine rules may be sub-optimal. In these
cases they serve as a tractable inner approximation.

**Current limitations** in discopt's implementation:
- Recourse variables must be scalar or 1-D.
- The policy matrix scales as $\dim(y) \times n_\xi$ new scalar variables,
  which can be large for high-dimensional problems.
- Non-affine recourse (polynomial, piecewise-linear) is not yet supported.

## References

```{bibliography}
:filter: docname in docnames
```